In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("../").resolve()
sys.path.append(str(PROJECT_ROOT))

from src.config import (
    BASE_DIR,
    TRAIN_PATH_STAGE2,
    TEST_PATH_STAGE2,
    CRAWLED_ABSTRACT_PATH,
    MERGED_TRAIN_PATH,
    MERGED_TEST_PATH,
    CLEANED_TRAIN_PATH,
    CLEANED_TEST_PATH
)

from src.utils.utils import load_csv, save_csv, preview_df

from src.preprocess.preprocess import (
    DataMerger,
    MissingHandler,
    AuthorNormalizer,
    TextCleaner
)

import pandas as pd

print("✔ Imports ready")

✔ Imports ready


In [2]:
train_df = load_csv(TRAIN_PATH_STAGE2)
test_df = load_csv(TEST_PATH_STAGE2)
crawl_df = load_csv(CRAWLED_ABSTRACT_PATH)

print("Train:", train_df.shape)
print("Test:", test_df.shape)
print("Crawl:", crawl_df.shape)

preview_df(train_df)

Train: (2494, 7)
Test: (596, 6)
Crawl: (3092, 6)
Shape: (2494, 7)
Columns: ['id', 'title', 'venue', 'year', 'authors', 'doi', 'Label']


,id,title,venue,year,authors,doi,Label
0,0,Proceedings 41st International Conference on L...,iclp,2026,Paul Tarau,https://www.semanticscholar.org/paper/f7391104...,1
1,1,Conditionals and Temporal Conditionals for Gra...,iclp,2025,NaN,https://www.semanticscholar.org/paper/8328d53d...,1
2,2,Learning and Contesting Assumption-based Argum...,iclp,2025,"Ofer Arieli, Jesse Heyninck",https://www.semanticscholar.org/paper/1c433ff0...,2
3,3,Agentified Argumentative Learning (short paper).,iclp,2025,NaN,https://www.semanticscholar.org/paper/666f0fa6...,1
4,4,Empowering Public Interest Communication with ...,iclp,2025,Alejandra Casas Niño de Rivera,https://www.semanticscholar.org/paper/08195948...,1


In [3]:
merger = DataMerger()

train_merged = merger.transform(train_df, crawl_df)
test_merged = merger.transform(test_df, crawl_df)

print("Train merged:", train_merged.shape)
print("Test merged:", test_merged.shape)

preview_df(train_merged)

Train merged: (2496, 10)
Test merged: (596, 9)
Shape: (2496, 10)
Columns: ['id', 'title', 'venue', 'year', 'authors', 'doi', 'Label', 'abstract', 'crawled_authors', 'source']


,id,title,venue,year,authors,doi,Label,abstract,crawled_authors,source
0,0,Proceedings 41st International Conference on L...,iclp,2026,Paul Tarau,https://www.semanticscholar.org/paper/f7391104...,1,Since the first conference in Marseille in 198...,"Martin Gebser, Daniela Inclezan, Francesco Ric...",SemanticScholarPaperIDCrawler
1,1,Conditionals and Temporal Conditionals for Gra...,iclp,2025,"Alexey Natekin, Alois Knoll",https://www.semanticscholar.org/paper/8328d53d...,1,Gradient boosting machines are a family of pow...,"Alexey Natekin, Alois Knoll",SemanticScholarPaperIDCrawler
2,2,Learning and Contesting Assumption-based Argum...,iclp,2025,"Ofer Arieli, Jesse Heyninck",https://www.semanticscholar.org/paper/1c433ff0...,2,Our goal in this article is to remind readers ...,"Eve Tuck, K. Wayne Yang",SemanticScholarPaperIDCrawler
3,3,Agentified Argumentative Learning (short paper).,iclp,2025,"Emanuele De Angelis, Maurizio Proietti, France...",https://www.semanticscholar.org/paper/666f0fa6...,1,Argumentative learning amounts to integrating ...,"Emanuele De Angelis, Maurizio Proietti, France...",SemanticScholarPaperIDCrawler
4,4,Empowering Public Interest Communication with ...,iclp,2025,Alejandra Casas Niño de Rivera,https://www.semanticscholar.org/paper/08195948...,1,The EPICA (Empowering Public Interest Communic...,"Pietro Baroni, Stefano Bistarelli, Bettina Faz...",SemanticScholarPaperIDCrawler


In [4]:
drop_cols = ["crawled_authors", "source"]

train_merged = train_merged.drop(

    columns=[c for c in drop_cols if c in train_merged.columns],

    errors="ignore"

)

test_merged = test_merged.drop(

    columns=[c for c in drop_cols if c in test_merged.columns],

    errors="ignore"

)

# =========================

# SAVE

# =========================

save_csv(train_merged, MERGED_TRAIN_PATH)

save_csv(test_merged, MERGED_TEST_PATH)

print("✔ Merged data saved to interim/")

Saved CSV -> /Users/nhatnam/Documents/DM_252/Assignment/data/interim/merged_train.csv
Saved CSV -> /Users/nhatnam/Documents/DM_252/Assignment/data/interim/merged_test.csv
✔ Merged data saved to interim/


In [5]:
missing = MissingHandler()

train_clean = missing.transform(train_merged)
test_clean = missing.transform(test_merged)

print("After missing handling:")
print(train_clean.isna().sum())

After missing handling:
id          0
title       0
venue       0
year        0
authors     0
doi         0
Label       0
abstract    0
dtype: int64


In [6]:
author_norm = AuthorNormalizer()

train_clean = author_norm.transform(train_clean)
test_clean = author_norm.transform(test_clean)

preview_df(train_clean)

Shape: (2496, 8)
Columns: ['id', 'title', 'venue', 'year', 'authors', 'doi', 'Label', 'abstract']


,id,title,venue,year,authors,doi,Label,abstract
0,0,Proceedings 41st International Conference on L...,iclp,2026,Paul Tarau,https://www.semanticscholar.org/paper/f7391104...,1,Since the first conference in Marseille in 198...
1,1,Conditionals and Temporal Conditionals for Gra...,iclp,2025,"Alexey Natekin, Alois Knoll",https://www.semanticscholar.org/paper/8328d53d...,1,Gradient boosting machines are a family of pow...
2,2,Learning and Contesting Assumption-based Argum...,iclp,2025,"Ofer Arieli, Jesse Heyninck",https://www.semanticscholar.org/paper/1c433ff0...,2,Our goal in this article is to remind readers ...
3,3,Agentified Argumentative Learning (short paper).,iclp,2025,"Emanuele De Angelis, Maurizio Proietti, France...",https://www.semanticscholar.org/paper/666f0fa6...,1,Argumentative learning amounts to integrating ...
4,4,Empowering Public Interest Communication with ...,iclp,2025,Alejandra Casas Niño de Rivera,https://www.semanticscholar.org/paper/08195948...,1,The EPICA (Empowering Public Interest Communic...


In [7]:
text_cleaner = TextCleaner()

train_clean = text_cleaner.transform(train_clean)
test_clean = text_cleaner.transform(test_clean)

preview_df(train_clean)

Shape: (2496, 8)
Columns: ['id', 'title', 'venue', 'year', 'authors', 'doi', 'Label', 'abstract']


,id,title,venue,year,authors,doi,Label,abstract
0,0,proceedings 41st international conference on l...,iclp,2026,Paul Tarau,https://www.semanticscholar.org/paper/f7391104...,1,since the first conference in marseille in 198...
1,1,conditionals and temporal conditionals for gra...,iclp,2025,"Alexey Natekin, Alois Knoll",https://www.semanticscholar.org/paper/8328d53d...,1,gradient boosting machines are a family of pow...
2,2,learning and contesting assumption-based argum...,iclp,2025,"Ofer Arieli, Jesse Heyninck",https://www.semanticscholar.org/paper/1c433ff0...,2,our goal in this article is to remind readers ...
3,3,agentified argumentative learning short paper,iclp,2025,"Emanuele De Angelis, Maurizio Proietti, France...",https://www.semanticscholar.org/paper/666f0fa6...,1,argumentative learning amounts to integrating ...
4,4,empowering public interest communication with ...,iclp,2025,Alejandra Casas Niño de Rivera,https://www.semanticscholar.org/paper/08195948...,1,the epica empowering public interest communica...


In [8]:
save_csv(train_clean, CLEANED_TRAIN_PATH)
save_csv(test_clean, CLEANED_TEST_PATH)

print("✔ Cleaned data saved")

Saved CSV -> /Users/nhatnam/Documents/DM_252/Assignment/data/interim/cleaned_train.csv
Saved CSV -> /Users/nhatnam/Documents/DM_252/Assignment/data/interim/cleaned_test.csv
✔ Cleaned data saved


In [9]:
print("FINAL SHAPES:")
print("Train:", train_clean.shape)
print("Test:", test_clean.shape)

train_clean.head()

FINAL SHAPES:
Train: (2496, 8)
Test: (596, 7)


,id,title,venue,year,authors,doi,Label,abstract
0,0,proceedings 41st international conference on l...,iclp,2026,Paul Tarau,https://www.semanticscholar.org/paper/f7391104...,1,since the first conference in marseille in 198...
1,1,conditionals and temporal conditionals for gra...,iclp,2025,"Alexey Natekin, Alois Knoll",https://www.semanticscholar.org/paper/8328d53d...,1,gradient boosting machines are a family of pow...
2,2,learning and contesting assumption-based argum...,iclp,2025,"Ofer Arieli, Jesse Heyninck",https://www.semanticscholar.org/paper/1c433ff0...,2,our goal in this article is to remind readers ...
3,3,agentified argumentative learning short paper,iclp,2025,"Emanuele De Angelis, Maurizio Proietti, France...",https://www.semanticscholar.org/paper/666f0fa6...,1,argumentative learning amounts to integrating ...
4,4,empowering public interest communication with ...,iclp,2025,Alejandra Casas Niño de Rivera,https://www.semanticscholar.org/paper/08195948...,1,the epica empowering public interest communica...


In [10]:
def missing_report(df, name="dataset", id_col="id"):
    print(f"\n===== MISSING REPORT: {name} =====")

    report_rows = []

    for col in df.columns:

        # =========================
        # Detect missing
        # =========================
        if df[col].dtype == "object":
            missing_mask = (
                df[col].isna()
                | (df[col].astype(str).str.strip() == "")
            )
        else:
            missing_mask = df[col].isna()

        missing_count = missing_mask.sum()
        missing_ratio = missing_count / len(df) * 100

        # =========================
        # Get missing IDs
        # =========================
        if missing_count > 0:

            if id_col in df.columns:
                missing_ids = df.loc[missing_mask, id_col].tolist()
            else:
                missing_ids = df.index[missing_mask].tolist()

            report_rows.append({
                "column": col,
                "missing_count": missing_count,
                "missing_ratio_%": round(missing_ratio, 2),
                "missing_ids": missing_ids
            })

    report = pd.DataFrame(report_rows)

    if len(report) == 0:
        print("No missing values 🎉")
    else:
        print(report)

    return report

# =========================
# RUN
# =========================
train_missing_report = missing_report(train_df, "RAW TRAIN")
test_missing_report = missing_report(test_df, "RAW TEST")

train_clean_missing_report = missing_report(train_clean, "CLEAN TRAIN")
test_clean_missing_report = missing_report(test_clean, "CLEAN TEST")


===== MISSING REPORT: RAW TRAIN =====
    column  missing_count  missing_ratio_%  \
0  authors            192              7.7   

                                         missing_ids  
0  [1, 3, 6, 10, 11, 12, 13, 15, 18, 21, 25, 37, ...  

===== MISSING REPORT: RAW TEST =====
    column  missing_count  missing_ratio_%  \
0  authors             41             6.88   

                                         missing_ids  
0  [1106, 1894, 1877, 1320, 1698, 1083, 2524, 121...  

===== MISSING REPORT: CLEAN TRAIN =====
No missing values 🎉

===== MISSING REPORT: CLEAN TEST =====
No missing values 🎉
